# Session 1 — Foundations & Your First API Call
**Date:** Sat 20 Jun 2026 | **Duration:** 4 hrs (9:30 am – 1:30 pm IST)  
**Course:** Python Applications with OpenAI — MSME Technology Development Center, Chennai

---

## Learning Objectives
By the end of this session you can:
- Use core Python constructs needed for AI apps: functions, dicts, JSON, env vars
- Explain the OpenAI API ecosystem and where Chat Completions fits in
- Set up a `.env` file, install the SDK, and authenticate safely
- Make a Chat Completions call and inspect the full response object
- Define token and estimate the cost of a request from `response.usage`

## Agenda
| Time | Topic |
|------|-------|
| 00:00 | Intro & environment check |
| 00:20 | Python essentials for AI apps |
| 01:00 | OpenAI ecosystem tour |
| 01:20 | **BREAK** |
| 01:30 | API key, billing & `.env` setup |
| 01:50 | First Chat Completions call |
| 02:20 | What is a token? |
| 02:40 | Reading usage + estimating cost |
| 03:00 | **BREAK** |
| 03:10 | Exercises & capstone kickoff |
| 03:40 | Recap & homework |

In [1]:
# ── Session 1 Setup (run this cell first — it must succeed before anything else) ──
import os
import json
import sys
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()  # loads OPENAI_API_KEY and MOCK from .env

# Locate project root (the directory that contains utils/) regardless of
# where jupyter was launched from.
_cwd = Path(os.getcwd())
_root = _cwd
for _ in range(5):
    if (_root / "utils").is_dir():
        break
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from utils.config import MODELS, get_client, estimate_cost
from utils.helpers import pretty_print, retry, mock_chat_response, token_cost_summary

MOCK  = os.getenv("MOCK", "0") == "1"   # set MOCK=1 in .env to skip real API calls
MODEL = MODELS["chat"]                   # gpt-5.4-mini — change to MODELS["frontier"] for gpt-5.5

# Smoke test
if MOCK:
    print(f"MOCK mode ON — no real API calls will be made. Model constant: {MODEL}")
else:
    try:
        client = get_client()
        models = client.models.list()
        print(f"Client is alive.  Default model: {MODEL}")
    except Exception as e:
        print(f"Setup issue: {e}\nCheck that OPENAI_API_KEY is set in your .env file.")

Setup issue: OPENAI_API_KEY not found. Copy .env.example → .env and paste your key, then re-run the setup cell.
Check that OPENAI_API_KEY is set in your .env file.


---
## 1. Python Essentials for AI Apps

Every OpenAI API request is a **Python dict** sent as JSON. Every response comes back as a structured Python object (or raw JSON). The three patterns you will use in *every* session:

1. **Functions with keyword arguments** — for building reusable call wrappers
2. **Nested dicts + JSON round-trip** — the exact shape of the API message format
3. **`os.getenv()`** — the only safe way to read your API key at runtime

We'll live-code all three now so they feel familiar before the first API call.

In [2]:
# ── 1a. Functions with keyword args ─────────────────────────────────────────
# GOAL: Write a helper that builds one OpenAI "message" dict
# STEPS:
#   1. Define build_message(role, content) -> dict
#   2. Call it: build_message(role="user", content="Hello!")
#   3. Print the returned dict
# --- live-code below ---

def build_message(role, content):
    return {"role": role, "content": content}   

build_message("user", "Hello!")




{'role': 'user', 'content': 'Hello!'}

In [ ]:
# ── 1b. Dicts, JSON round-trip, and env vars ────────────────────────────────
# GOAL: See the exact JSON shape we send to the API, and safely peek at the key
# STEPS:
#   1. Build a messages list: one system dict + one user dict
#   2. json.dumps(messages, indent=2) and print — this is what travels over the wire
#   3. key = os.getenv("OPENAI_API_KEY", "")
#      print(f"Key loaded: {bool(key)} — starts with: {key[:8]}...")  # never print full key!
# --- live-code below ---

messages = [
    build_message("system", "You are a helpful assistant."),
    build_message("user", "What is the capital of India?")
]

json.dumps(messages, indent=2)








'[\n  {\n    "role": "system",\n    "content": "You are a helpful assistant."\n  },\n  {\n    "role": "user",\n    "content": "What is the capital of India?"\n  }\n]'

---
## 2. OpenAI Ecosystem Tour

> **Note:** The course flyer uses 2023 product names. Several have been renamed, replaced, or retired. This section maps flyer terms to what exists today (June 2026).

### Two main API interfaces
| Interface | Use it for | Status |
|-----------|-----------|--------|
| **Chat Completions API** | Simple request → response; easiest to learn | Stable, ubiquitous |
| **Responses API** | Agentic loops, built-in tools (web_search, file_search, code_interpreter, image_generation) | Modern; replaces the deprecated Assistants API |

We teach **Chat Completions first** (Sessions 1–3), then move to the **Responses API** in Session 3.

### Model families (June 2026)
| Tier | Models | Best for |
|------|--------|----------|
| Ultra-cheap | `gpt-5.4-nano` | High-volume, low-stakes tasks |
| Workhorse | **`gpt-5.4-mini`** ← default | Everyday demos; this course |
| Standard | `gpt-5.4` | Balanced quality/cost |
| Frontier | `gpt-5.5`, `gpt-5.5-pro` | Hard reasoning; use sparingly |
| Reasoning | `o4-mini` | Multi-step logical problems |
| Image | `gpt-image-1` | Image generation (DALL·E 3 retired May 2026) |
| Audio | `gpt-4o-mini-transcribe`, `gpt-4o-mini-tts` | STT / TTS |

All model names in this course live in `utils/config.py::MODELS` — never a bare string in a notebook cell.

In [ ]:
# ── 2. Explore the MODELS config ────────────────────────────────────────────
# GOAL: Inspect the central config and understand why we never hardcode model names
# STEPS:
#   1. Print MODELS (already imported from utils.config)
#   2. Print MODEL (the session default set in the setup cell)
#   3. Discussion: what would break if we hardcoded "gpt-5.4-mini" in 30 notebook cells
#      and the model name changed tomorrow?
# --- live-code below ---


---
## 3. API Key, Billing & `.env` Setup

### Non-negotiable key safety rules
1. **Never hardcode** an API key in a notebook, script, or commit.
2. Store it in a `.env` file at the project root and load it with `python-dotenv`.
3. `.env` is already in `.gitignore` — confirm this before your first `git push`.
4. In the OpenAI dashboard → **Billing → Usage limits**, set a monthly cap before running demos (₹500–₹1 000 / ~$5–$10 is enough for this course).

### `.env` file format
```
OPENAI_API_KEY=sk-proj-...
MOCK=0
```
Copy `.env.example` → `.env` and paste your key. The setup cell above calls `load_dotenv()` which reads this file automatically.

### Billing context for this session
All exercises in Session 1 use `gpt-5.4-mini` (~$0.00015 per 1 K input tokens). The total spend for this session is **< $0.05** if you follow the exercises as written. Setting `MOCK=1` in `.env` costs nothing at all.

In [ ]:
# ── 3. Verify your API key is loaded ────────────────────────────────────────
# GOAL: Confirm the key exists in the environment without printing it in full
# STEPS:
#   1. key = os.getenv("OPENAI_API_KEY", "")
#   2. print(f"Key loaded: {bool(key)}")
#   3. If key: print(f"Starts with: {key[:8]}...")  # safe preview
#   4. Else: print a clear instruction to fix .env
# --- live-code below ---


---
## 4. Your First Chat Completions Call

The Chat Completions API takes a **list of messages** — each a dict with `role` and `content` — and returns a structured response.

### Message roles
| Role | Purpose |
|------|---------|
| `system` | Sets the model's persona/context for the whole conversation |
| `user` | What the human says |
| `assistant` | What the model replied (used to build multi-turn history — Session 2) |

### Minimal call pattern
```python
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user",   "content": "Hello!"}
    ]
)
print(response.choices[0].message.content)
```
The response is a `ChatCompletion` object. The text lives at `response.choices[0].message.content`.

In [ ]:
# ── 4a. Make the first API call ──────────────────────────────────────────────
# GOAL: Send a single-turn message and print the reply
# STEPS:
#   1. Build messages: system = "You are a helpful assistant for Indian professionals."
#                      user   = "In one sentence, what is the OpenAI API?"
#   2. if MOCK:
#          response = mock_chat_response("The OpenAI API lets you call AI models programmatically.")
#      else:
#          response = client.chat.completions.create(model=MODEL, messages=messages)
#   3. print(response.choices[0].message.content)
# --- live-code below ---


In [ ]:
# ── 4b. Inspect the full response object ────────────────────────────────────
# GOAL: Understand what the API actually returns beyond the text
# STEPS:
#   1. print(type(response))                         # ChatCompletion
#   2. print("Model:", response.model)               # actual model used
#   3. print("Finish reason:", response.choices[0].finish_reason)  # "stop" = normal
#   4. print("Usage:", response.usage)               # prompt + completion tokens
#   5. pretty_print(response)                        # from utils.helpers
# --- live-code below ---


---
## 5. What Is a Token?

The API bills by **token**, not by character or word.

- 1 token ≈ 4 characters in English (≈ 0.75 words)
- `"Hello, world!"` ≈ 4 tokens
- Both **input tokens** (your messages) and **output tokens** (the reply) are charged
- Tokens are model-specific; **`tiktoken`** is OpenAI's open-source tokenizer library

### Why this matters in practice
- A long `system` prompt costs money on *every* call — keep it tight
- Structured JSON outputs with many keys are token-heavy
- `max_tokens` caps the completion length (and therefore cost)
- In Session 3 we'll stream tokens as they arrive in real time

In [ ]:
# ── 5. Tokenise text with tiktoken ──────────────────────────────────────────
# GOAL: See how text is split into tokens
# STEPS:
#   1. import tiktoken
#   2. enc = tiktoken.encoding_for_model("gpt-4o")  # closest public encoding
#   3. text = "Hello, I am a student at MSME-TDC Chennai."
#      tokens = enc.encode(text)
#      print(f"Text: {text!r}")
#      print(f"Token count: {len(tokens)}")
#      print(f"Tokens: {tokens}")
#   4. Repeat with a longer paragraph and observe how count scales
# --- live-code below ---


---
## 6. Reading Usage & Estimating Cost

Every `ChatCompletion` response carries a `usage` object:

```python
response.usage.prompt_tokens      # tokens you sent (system + user messages)
response.usage.completion_tokens  # tokens the model generated
response.usage.total_tokens       # sum of both
```

### Approximate prices (June 2026 — always verify at openai.com/pricing)
| Model | Input / 1K tokens | Output / 1K tokens |
|-------|------------------|--------------------|
| `gpt-5.4-nano` | ~$0.00005 | ~$0.00020 |
| `gpt-5.4-mini` | ~$0.00015 | ~$0.00060 |
| `gpt-5.5` | ~$0.00300 | ~$0.01200 |

`utils/config.py::estimate_cost(usage, model)` wraps this calculation for you.

In [ ]:
# ── 6. Print usage and estimate cost ────────────────────────────────────────
# GOAL: Turn response.usage into a human-readable cost estimate
# STEPS:
#   1. u = response.usage
#      print(f"Prompt tokens:     {u.prompt_tokens}")
#      print(f"Completion tokens: {u.completion_tokens}")
#      print(f"Total tokens:      {u.total_tokens}")
#   2. cost = estimate_cost(response.usage, MODEL)
#      print(f"Estimated cost: ${cost:.6f}")
#   3. Compare: what would the same call cost on gpt-5.5?
#      cost_frontier = estimate_cost(response.usage, MODELS["frontier"])
#      print(f"Same call on gpt-5.5: ${cost_frontier:.6f}")
# --- live-code below ---


---
## Exercises

> **Exercise 1 — Build a reusable `chat()` helper**
>
> Write a function `chat(prompt, system="You are a helpful assistant.", **kwargs) -> str` that:
> 1. Builds the messages list
> 2. Returns a mock string if `MOCK=1`, otherwise calls the API
> 3. Returns `response.choices[0].message.content`
>
> Test it: `print(chat("Name one use of AI in Indian manufacturing."))`

In [ ]:
# your turn:


> **Exercise 2 — Token budget experiment**
>
> Call your `chat()` helper passing `max_tokens=20`.
> 1. What does `finish_reason` show when the model hits the token limit? (`"stop"` vs `"length"`)
> 2. How does the cost compare to an uncapped call?
> 3. Try `max_tokens=5` — does the reply make sense? Why/why not?

In [ ]:
# your turn:


> **Exercise 3 — Temperature demo**
>
> Call `chat()` three times with the **same prompt** and `temperature=0`, `0.7`, `1.2`.
> Print each reply. What do you observe about consistency and creativity?

In [ ]:
# your turn:


---
## Common Pitfalls

| Error | Likely cause | Fix |
|-------|-------------|-----|
| `AuthenticationError` | Key missing, wrong, or `load_dotenv()` not called | Check `.env`, confirm setup cell ran first |
| `NotFoundError` / model not found | Hardcoded or misspelled model string | Always use `MODELS["chat"]` from config |
| `RateLimitError` | Too many requests or free-tier throttling | Add `retry()` from `utils.helpers`, or switch to `MOCK=1` |
| `json.JSONDecodeError` | Calling `json.loads()` on a string that isn't JSON | Only parse when you explicitly asked the model for JSON output |
| Key visible in output/git | Printed or committed `.env` | Rotate the key immediately; never print more than the first 8 chars |
| `ModuleNotFoundError: utils` | Jupyter launched from the wrong directory | The setup cell's `sys.path` loop handles this — re-run the setup cell |

---
## Recap

- The OpenAI Chat Completions API takes a **list of message dicts** (`role` + `content`) and returns a `ChatCompletion` object.
- Always load secrets from `.env` with `python-dotenv` — never hardcode keys.
- All model names live in `utils/config.py::MODELS`; one edit re-points the entire course.
- Tokens are the billing unit: inspect `response.usage` and call `estimate_cost()` to stay on budget.
- `MOCK=1` in `.env` lets every notebook run fully offline without consuming API credits.

---
## Homework / Capstone Increment

Before Session 2 (Sun 21 Jun):

1. **Billing check:** Log into [platform.openai.com](https://platform.openai.com) → Usage. Confirm you can see today's spend from this session.

2. **Extend `chat()`:** Add a `temperature` parameter (default `0.7`) and a `max_tokens` parameter (default `None`). Make sure they pass through to the API call via `**kwargs` or explicit params.

3. **Capstone thread — start here:** Choose a domain you work in (finance, manufacturing, healthcare, retail, logistics…). Write a `system` message that gives the model a useful expert persona for that domain. Save it in a variable called `MY_SYSTEM`. We'll reuse and extend this across all six sessions to build your **personal AI assistant** incrementally.

4. **Optional:** Read the OpenAI pricing page and note the per-token cost of `gpt-5.4-mini` vs `gpt-5.5`. We'll use this comparison when we introduce the Responses API in Session 3.